# D4-v5 Signal Generalization Test

**Objective**: Verify that D4-v5 (10 signals) generalizes across:
- Format: EPSRC outlines (~400-700 words) vs NIH full proposals (~6 pages)
- Domain: AI/math vs biomedical vs legal/policy

**Pass criteria**:
1. No parsing failures (all signals return valid scores)
2. Aggregate range ≥ 0.15 (sufficient discrimination)
3. No signal at floor (1) or ceiling (5) on ALL proposals
4. EPSRC and NIH distributions overlap reasonably
5. No signal pair with r > 0.90 (redundancy)

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path(".")
SIGNAL_IDS = [
    "G1_problem_specificity", "G2_specific_aims", "G3_technical_evidence",
    "G4_named_methods", "G5_gap_identification", "G6_novel_contribution",
    "G8_deliverable_clarity", "G9_scope_feasibility",
    "G10_approach_coverage", "G11_preliminary_evidence",
]
SHORT_IDS = [s.split("_", 1)[0] for s in SIGNAL_IDS]

In [ ]:
def load_results(corpus: str) -> pd.DataFrame:
    path = RESULTS_DIR / f"results_{corpus}.jsonl"
    rows = []
    with open(path) as f:
        for line in f:
            entry = json.loads(line)
            row = {"name": entry["name"], "corpus": entry["corpus"],
                   "word_count": entry["word_count"], "aggregate": entry["aggregate"]}
            for sid in SIGNAL_IDS:
                row[sid] = entry["scores"].get(sid)
            rows.append(row)
    return pd.DataFrame(rows)

df_epsrc = load_results("epsrc")
df_nih = load_results("nih")
df = pd.concat([df_epsrc, df_nih], ignore_index=True)
print(f"Total proposals: {len(df)} (EPSRC: {len(df_epsrc)}, NIH: {len(df_nih)})")
df.head()

## 1. Parsing Check
All signals should return non-null scores.

In [ ]:
null_counts = df[SIGNAL_IDS].isnull().sum()
print("Null counts per signal:")
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "All signals parsed successfully!")

## 2. Per-Signal Statistics (EPSRC vs NIH)

In [ ]:
stats = []
for corpus_name, corpus_df in [("EPSRC", df_epsrc), ("NIH", df_nih), ("ALL", df)]:
    for sid, short in zip(SIGNAL_IDS, SHORT_IDS):
        vals = corpus_df[sid].dropna()
        stats.append({
            "corpus": corpus_name, "signal": short,
            "mean": vals.mean(), "std": vals.std(),
            "min": vals.min(), "max": vals.max(),
            "n": len(vals),
        })

stats_df = pd.DataFrame(stats)
pivot = stats_df.pivot_table(index="signal", columns="corpus", values="mean")
pivot = pivot[["EPSRC", "NIH", "ALL"]]
print("\nPer-signal mean scores:")
print(pivot.round(2).to_string())

## 3. Floor/Ceiling Check
No signal should be 1 or 5 on ALL proposals.

In [ ]:
print("Floor check (signal=1 on ALL proposals):")
for sid, short in zip(SIGNAL_IDS, SHORT_IDS):
    vals = df[sid].dropna()
    if (vals == 1).all():
        print(f"  FAIL: {short} is 1 on all {len(vals)} proposals")
    elif (vals == 5).all():
        print(f"  FAIL: {short} is 5 on all {len(vals)} proposals")
print("  (No floor/ceiling failures detected)" if True else "")

print("\nSignals with low variance (std < 0.5):")
for sid, short in zip(SIGNAL_IDS, SHORT_IDS):
    vals = df[sid].dropna()
    if vals.std() < 0.5:
        print(f"  WARNING: {short} std={vals.std():.2f} mean={vals.mean():.1f} — low discrimination")

## 4. Aggregate Score Distribution

In [ ]:
for corpus_name, corpus_df in [("EPSRC", df_epsrc), ("NIH", df_nih), ("ALL", df)]:
    agg = corpus_df["aggregate"]
    print(f"{corpus_name:6s}: mean={agg.mean():.3f}  std={agg.std():.3f}  "
          f"range=[{agg.min():.3f}, {agg.max():.3f}]  spread={agg.max()-agg.min():.3f}")

agg_range = df["aggregate"].max() - df["aggregate"].min()
print(f"\nOverall aggregate range: {agg_range:.3f} {'PASS (≥0.15)' if agg_range >= 0.15 else 'FAIL (<0.15)'}")

## 5. Inter-Signal Correlation Matrix

In [ ]:
corr = df[SIGNAL_IDS].corr()
corr.columns = SHORT_IDS
corr.index = SHORT_IDS
print("Correlation matrix:")
print(corr.round(2).to_string())

print("\nHigh correlations (|r| > 0.70):")
for i, si in enumerate(SIGNAL_IDS):
    for j, sj in enumerate(SIGNAL_IDS):
        if i < j:
            r = corr.iloc[i, j]
            if abs(r) > 0.70:
                print(f"  {SHORT_IDS[i]} × {SHORT_IDS[j]}: r={r:.2f}"
                      f" {'FAIL (>0.90)' if abs(r) > 0.90 else 'WARNING'}")

## 6. Score Heatmap

In [ ]:
display_df = df[["name", "corpus"] + SIGNAL_IDS + ["aggregate"]].copy()
display_df["name"] = display_df["name"].str[:40]
display_df.columns = ["name", "corpus"] + SHORT_IDS + ["agg"]
print(display_df.to_string(index=False))

## 7. Summary & Go/No-Go

In [ ]:
checks = {
    "No parsing failures": df[SIGNAL_IDS].isnull().sum().sum() == 0,
    "Aggregate range ≥ 0.15": (df["aggregate"].max() - df["aggregate"].min()) >= 0.15,
    "No floor signal (all=1)": not any((df[sid].dropna() == 1).all() for sid in SIGNAL_IDS),
    "No ceiling signal (all=5)": not any((df[sid].dropna() == 5).all() for sid in SIGNAL_IDS),
    "No redundant pair (r>0.90)": not any(
        abs(corr.iloc[i, j]) > 0.90
        for i in range(len(SIGNAL_IDS)) for j in range(i+1, len(SIGNAL_IDS))
    ),
}

print("=" * 50)
print("D4-v5 Generalization Test: Go/No-Go")
print("=" * 50)
all_pass = True
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}] {check}")
print(f"\nOverall: {'GO — proceed to CR-v7 pipeline' if all_pass else 'NO-GO — investigate failures'}")